## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
date = '20241101'

In [9]:
## base

base_query = f"""

select 
    eventname,
    yyyymmdd,
    hhmmss,
    quarter_hour,
    epoch,
    profile_identity,
    profile_phone,
    eventProps,
    json_extract_scalar(eventProps, '$.eventProps.currentCity') currentCity
from 
    raw.clevertap_customer_events_master 
where 
    yyyymmdd='{date}'
    and quarter_hour in ('1000', '1015', '1030', '1045', '1100')
    and profile_platform = 'Android'
    and json_extract_scalar(eventProps, '$.eventProps.currentCity') = 'Bangalore'
    -- and not regexp_like (eventname, 'payment|upi_|web_|c2c')
"""

df_base_query = pd.read_sql(base_query, connection)
df_base_query

,eventname,yyyymmdd,hhmmss,quarter_hour,epoch,profile_identity,profile_phone,eventProps,currentCity
0,widget_action_failures,20241101,104659,1045,1730438219000,656f2836e608410c3a4af804,919741204626,"{""eventProps"":{""cleverTapId"":""__6e1aba591fd444...",Bangalore
1,widget_action_failures,20241101,100950,1000,1730435990000,64fbe1806cd001b5fa193282,918639441540,"{""eventProps"":{""cleverTapId"":""__1b542fd4451147...",Bangalore
2,widget_action_failures,20241101,103707,1030,1730437627000,64d311484ba535dc1c9cef59,919792280926,"{""eventProps"":{""cleverTapId"":""__f15e54febb3e4c...",Bangalore
3,widget_action_failures,20241101,101836,1015,1730436516000,671f552734b572429ff5288c,918473849557,"{""eventProps"":{""cleverTapId"":""__6536954dbf5c4d...",Bangalore
4,widget_action_failures,20241101,101932,1015,1730436572000,625ce1911194b4f520e96d5f,919742510739,"{""eventProps"":{""cleverTapId"":""__a52295b2728943...",Bangalore
...,...,...,...,...,...,...,...,...,...
1341305,fareestimate,20241101,111456,1100,1730439896000,61dbb327d80f2b0cb4a38cd0,919663737906,"{""eventProps"":{""HFDistance"":29684.752,""bidEnab...",Bangalore
1341306,fareestimate,20241101,111457,1100,1730439897000,61dbb327d80f2b0cb4a38cd0,919663737906,"{""eventProps"":{""HFDistance"":29684.752,""bidEnab...",Bangalore
1341307,fareestimate,20241101,111457,1100,1730439897000,61dbb327d80f2b0cb4a38cd0,919663737906,"{""eventProps"":{""HFDistance"":29684.752,""bidEnab...",Bangalore
1341308,fareestimate,20241101,111457,1100,1730439897000,61dbb327d80f2b0cb4a38cd0,919663737906,"{""eventProps"":{""HFDistance"":29684.752,""bidEnab...",Bangalore


In [10]:
df_base_query.shape

(1341310, 9)

In [11]:
df_base_query.to_parquet('data_dump_for_process_mining_bangalore.parquet', index=False)

In [12]:
test1 = pd.read_parquet('data_dump_for_process_mining_without_payment.parquet')

In [13]:
test1.shape

(1338288, 9)

In [15]:
1341310-1338288

3022

In [18]:
## base

base_query = f"""

SELECT
    city AS city,
    pickup_cluster,
    user_id as customer_id,
    fare_estimate_id,service_detail_id as service_id_fare,
    api_context,
    date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
    cast(final_amount AS double) AS fe_amount,polyline,
    epoch
FROM
    hive.pricing.fare_estimates_enriched
WHERE
     yyyymmdd = '20241021'             
     and service_name in ('Link')
     AND city = 'Bangalore'
     AND api_context='/fare/estimate'
"""

df_base_query = pd.read_sql(base_query, connection)
df_base_query

DatabaseError: {'message': 'Query exceeded the maximum execution time limit of 10.00m', 'errorCode': 131075, 'errorName': 'EXCEEDED_TIME_LIMIT', 'errorType': 'INSUFFICIENT_RESOURCES', 'failureInfo': {'type': 'io.trino.spi.TrinoException', 'message': 'Query exceeded the maximum execution time limit of 10.00m', 'suppressed': [], 'stack': ['io.trino.execution.QueryTracker.enforceTimeLimits(QueryTracker.java:187)', 'io.trino.execution.QueryTracker.lambda$start$0(QueryTracker.java:89)', 'com.google.common.util.concurrent.MoreExecutors$ScheduledListeningDecorator$NeverSuccessfulListenableFutureTask.run(MoreExecutors.java:730)', 'java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)', 'java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)', 'java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)', 'java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.jav

In [ ]:
df_base_query.to_parquet('fare_estimates_bangalore_link_20241021.parquet', index=False)